# 03 - Embedding and retrieval

**Goal.** Embed the chunks of the sample documents, index them, and run retrieval: the sparse lexical
encoder, hybrid dense + sparse search with reciprocal rank fusion (RRF), reranking, tenant isolation
and deletion.

**What it checks** (each ends with an `assert`):
- `LexicalSparseEncoder` gives deterministic sparse vectors; Urdu digits (`۴۵۰۰`) hit the same index as ASCII (`4500`)
- hybrid search returns RRF-fused hits with dense similarity, sparse score and RRF score, best match first
- the same query in English and Urdu fuse across variants (`fuse_rrf`), and the reranker orders the candidates
- metadata filters narrow the search (a province filter returns only that province)
- tenant isolation: another user's private document is never returned
- deleting a document removes its points

**Modes.** If `OPENAI_API_KEY` (or `LLM_API_KEY`) is set (environment or `.env`, read through the app's `Settings`, never printed),
the project's factory (`build_retrieval`) provides the OpenAI embedder, the local reranker and a Qdrant
store (in-memory Qdrant). Otherwise it runs offline with the deterministic `FakeEmbedder` dense vectors,
the project's `LexicalSparseEncoder` for the sparse half (exactly how the hosted embedder composes them),
`FakeReranker` and the in-memory store from `tests/fakes.py`. Set `RAG_NOTEBOOKS_OFFLINE=1` to force offline.

**How to run.** From the repo root: `.venv/Scripts/python.exe scripts/run_notebooks.py 03`.

In [1]:
import warnings

warnings.filterwarnings("ignore")
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "app").is_dir() else Path.cwd().parent
sys.path[:0] = [str(ROOT), str(ROOT / "notebooks")]

from _support import ALICE, BOB, banner, build_stack, clip, index_samples, make_samples, new_workdir
from app.protocols import RetrievalQuery, SearchFilters
from app.retrieval.hosted import LexicalSparseEncoder
from app.retrieval.retriever import fuse_rrf

print(banner())
stack = build_stack()
await stack.astart()
settings = stack.settings
print("mode     :", stack.mode)
print("embedder :", stack.embedder.model_name, f"(dimension {stack.embedder.dimension})")
print("reranker :", stack.reranker.model_name)
print("store    :", type(stack.store).__name__)

 offline mode: fake embeddings - run with an OpenAI key for real scores
mode     : offline
embedder : fake-hash-dense+lexical-sparse (dimension 64)
reranker : fake-overlap-reranker
store    : InMemoryVectorStore


## 1. The sparse lexical encoder
Tokens are normalised (NFKC, Urdu letters unified, Urdu/Arabic-Indic digits to ASCII, diacritics
stripped, case-folded), stopwords dropped, and each token hashed to a stable 31-bit index. Queries weigh a
token 1.0; chunks use BM25-style saturation. There is no IDF, on purpose (an IDF over the whole
collection would leak what other tenants uploaded).

In [2]:
enc = LexicalSparseEncoder()


def token_table(text: str, vector) -> dict[str, float]:
    """token -> weight, recovered by hashing each token (the encoder only stores indices)."""
    by_index = {i: w for i, w in zip(vector.indices, vector.values)}
    out = {}
    for tok in sorted(set(enc._tokens(text))):
        idx = enc.encode_query(tok).indices[0]
        out[tok] = round(by_index[idx], 2)
    return out


english = "What is the urgent passport fee for 4500 rupees?"
urdu = "فوری پاسپورٹ کی فیس ۴۵۰۰ روپے کتنی ہے؟"
v_en, v_ur = enc.encode_query(english), enc.encode_query(urdu)
print("EN :", token_table(english, v_en))
print("UR :", token_table(urdu, v_ur))
print("nonzero entries:", len(v_en.indices), "(EN),", len(v_ur.indices), "(UR)")

# deterministic across calls, and Urdu digits land on the same index as ASCII digits
assert enc.encode_query(english) == v_en
idx_ascii = enc.encode_query("4500").indices
assert enc.encode_query("۴۵۰۰").indices == idx_ascii, "Urdu digits must equal ASCII digits"
assert enc.encode_query("٤٥٠٠").indices == idx_ascii, "Arabic-Indic digits too"
assert idx_ascii[0] in v_en.indices and idx_ascii[0] in v_ur.indices
print("digit '4500' index", idx_ascii[0], "appears in both the English and the Urdu query vector")
d = enc.encode_document("Normal passport fee is Rs 4500. The fee is paid at the bank. The fee is final.")
print("document weights are saturated (the term 'fee' occurs 3 times):", dict(zip(d.indices[:3], [round(x, 2) for x in d.values[:3]])))
assert max(d.values) < 3.0

# Note: numbers with a thousands separator are split into two tokens, so "4,500" in a chunk does not
# share a sparse index with the query token "4500" (the dense side may still match).
print("tokens of 'Rs 4,500':", enc._tokens("Rs 4,500"), "| tokens of 'Rs 4500':", enc._tokens("Rs 4500"))

EN : {'4500': 1.0, 'fee': 1.0, 'passport': 1.0, 'rupees': 1.0, 'urgent': 1.0}
UR : {'4500': 1.0, 'روپے': 1.0, 'فوری': 1.0, 'فیس': 1.0, 'پاسپورٹ': 1.0, 'کتنی': 1.0}
nonzero entries: 5 (EN), 6 (UR)
digit '4500' index 1331083772 appears in both the English and the Urdu query vector
document weights are saturated (the term 'fee' occurs 3 times): {118055006: 1.65, 599882972: 1.65, 677767027: 1.65}
tokens of 'Rs 4,500': ['rs', '4500'] | tokens of 'Rs 4500': ['rs', '4500']


## 2. Embed and index the sample documents
The chunks come from the same code as notebook 02. Each chunk is embedded with its context header
(`agency (jurisdiction) > title > section`) prepended, and stored with a deterministic point id.
One document (`helpdesk_note.txt`) belongs to a different user (`u_bob`); the others are public.

In [3]:
work = new_workdir()
samples = make_samples(work)
KEYS = ["pdf_table", "pdf_pages", "docx", "pptx", "html", "md", "urdu", "private"]
doc_ids = await index_samples(samples, KEYS, settings, stack.embedder, stack.store)
print(f"{'doc':10} {'owner':9} chunks")
total = 0
for key, doc_id in doc_ids.items():
    owner = samples[key].owner
    reader = BOB if owner == BOB.user_id else ALICE
    n = await stack.store.count(reader, doc_id)
    total += n
    print(f"{key:10} {owner:9} {n:>3}")
    assert n > 0
e = await stack.embedder.embed_query("passport fee")
print("query embedding: dense dim", len(e.dense), "| sparse entries", len(e.sparse.indices))
assert len(e.dense) == stack.embedder.dimension

doc        owner     chunks
pdf_table  u_public    2
pdf_pages  u_public    1
docx       u_public    6
pptx       u_public    3
html       u_public    4
md         u_public    3
urdu       u_public    1
private    u_bob       1
query embedding: dense dim 64 | sparse entries 2


## 3. Hybrid search with RRF
`hybrid_search` ranks by dense similarity and by sparse score separately, then fuses the two rankings
with RRF: `score = sum(1 / (60 + rank))`. A chunk ranked high by both wins even if neither score is the top one.

In [4]:
SEARCH = dict(prefetch_k=settings.prefetch_k, limit=5)


def show_hits(hits, title: str = "") -> None:
    print(title)
    print(f"{'#':>2} {'rrf':>7} {'dense':>7} {'sparse':>7}  {'file':24} text")
    for i, h in enumerate(hits, 1):
        sparse = f"{h.sparse_score:.2f}" if h.sparse_score is not None else "-"
        print(f"{i:>2} {h.rrf_score or 0:>7.4f} {h.similarity:>7.3f} {sparse:>7}  {h.payload.file_name:24.24} {clip(h.payload.text, 48)!r}")


question = "urgent passport fee for 36 pages"
q = await stack.embedder.embed_query(question)
hits = await stack.store.hybrid_search(ALICE, q, filters=SearchFilters(), **SEARCH)
show_hits(hits, f"query: {question!r}")
assert hits and all(h.rrf_score and h.rrf_score > 0 for h in hits)
assert [h.rrf_score for h in hits] == sorted((h.rrf_score for h in hits), reverse=True)
assert any("passport" in h.payload.file_name for h in hits[:3]), "a passport document should rank in the top 3"
best = hits[0]
print("\nbest hit:", best.payload.file_name, "| section:", best.payload.section_path or "-", "| page:", best.payload.page_start)

query: 'urgent passport fee for 36 pages'
 #     rrf   dense  sparse  file                     text
 1  0.0325   0.421    6.87  passport_fee_table.pdf   '| Category | Pages | Fee | |---|---|---| | Norm…'
 2  0.0325   0.377    8.36  passport_guide.pdf       'Immigration and Passports: Machine Readable Pas…'
 3  0.0308   0.188    3.08  nadra_cnic_guide.docx    'Fees The fee for the normal renewal category is…'
 4  0.0308   0.190    1.85  passport_fee_table.pdf   'Immigration and Passports fee table. Date of is…'
 5  0.0306   0.292    0.00  nadra_cnic_guide.docx    'Eligibility NADRA issues the computerised natio…'

best hit: passport_fee_table.pdf | section: - | page: 1


### An Urdu query, and fusion across query variants
The ask pipeline searches each query variant (original, English, Urdu) and fuses the lists with `fuse_rrf`.
The Urdu query below contains Urdu digits; the sparse side matches `4500` in the Urdu document.

In [5]:
urdu_q = "عام پاسپورٹ کی فیس ۴۵۰۰ روپے"
hits_ur = await stack.store.hybrid_search(ALICE, await stack.embedder.embed_query(urdu_q), filters=SearchFilters(), **SEARCH)
show_hits(hits_ur, f"query: {urdu_q!r}")
urdu_hit = next((h for h in hits_ur[:3] if h.payload.file_name == "passport_fee_urdu.txt"), None)
assert urdu_hit is not None, "the Urdu passport document should be in the top 3"
assert urdu_hit.sparse_score and urdu_hit.sparse_score > 0, "matched through the sparse (lexical) side"

fused = fuse_rrf([list(hits), list(hits_ur)])
print()
show_hits(fused[:5], "fused over the English and Urdu variants (cross-variant RRF):")
assert fused[0].rrf_score >= fused[-1].rrf_score
assert len({h.point_id for h in fused}) == len(fused), "one entry per point after fusion"

query: 'عام پاسپورٹ کی فیس ۴۵۰۰ روپے'
 #     rrf   dense  sparse  file                     text
 1  0.0328   0.699    8.90  passport_fee_urdu.txt    'پاسپورٹ کی فیس عام پاسپورٹ کی فیس ۴۵۰۰ روپے ہے۔…'
 2  0.0323   0.361    1.59  passport_fee_table.pdf   '| Category | Pages | Fee | |---|---|---| | Norm…'
 3  0.0317   0.286    1.32  passport_guide.pdf       'Immigration and Passports: Machine Readable Pas…'
 4  0.0301   0.190    0.00  passport_fee_table.pdf   'Immigration and Passports fee table. Date of is…'
 5  0.0297   0.244    0.00  fbr_tax_slides.pptx      'FBR income tax filing Filers submit the annual …'

fused over the English and Urdu variants (cross-variant RRF):
 #     rrf   dense  sparse  file                     text
 1  0.0325   0.421    6.87  passport_fee_table.pdf   '| Category | Pages | Fee | |---|---|---| | Norm…'
 2  0.0320   0.377    8.36  passport_guide.pdf       'Immigration and Passports: Machine Readable Pas…'
 3  0.0312   0.190    1.85  passport_fee_table.pdf   'I

### Rerank and pick the context
`HybridRetriever` embeds each variant, searches, fuses, dedupes, reranks the candidates against the
original question and keeps the top chunks above `tau_keep` as the LLM context.

In [6]:
result = await stack.retriever.retrieve(
    ALICE, RetrievalQuery(variants=(question,), candidates_k=settings.candidates_k, context_k=settings.context_k)
)
print(f"candidates={len(result.candidates)} context={len(result.context)} max_rerank={result.max_rerank:.3f} latency_ms={result.latency_ms}")
print(f"{'rank':>4} {'rerank':>7} {'ctx':>3}  {'file':24} text")
for c in result.candidates[:5]:
    print(f"{c.rank:>4} {c.rerank_score:>7.3f} {'*' if c.used_in_context else ' ':>3}  {c.point.payload.file_name:24.24} {clip(c.point.payload.text, 48)!r}")
assert result.context and result.max_rerank is not None
assert [c.rerank_score for c in result.candidates] == sorted((c.rerank_score for c in result.candidates), reverse=True)
assert all(c.rerank_score >= settings.tau_keep for c in result.context[1:])

candidates=20 context=4 max_rerank=1.000 latency_ms={'embed': 0, 'search': 1, 'rerank': 0, 'total': 3}
rank  rerank ctx  file                     text
   1   1.000   *  passport_guide.pdf       'Immigration and Passports: Machine Readable Pas…'
   2   0.667   *  passport_fee_table.pdf   '| Category | Pages | Fee | |---|---|---| | Norm…'
   3   0.500   *  nadra_cnic_guide.docx    'Fees The fee for the normal renewal category is…'
   4   0.333   *  nadra_cnic_guide.docx    '| Category | Processing time | Fee | |---|---|-…'
   5   0.167      passport_fee_table.pdf   'Immigration and Passports fee table. Date of is…'


## 4. Metadata filters
The ask pipeline narrows by agency and province when the question names them. Here a `kp` filter
keeps only the Khyber Pakhtunkhwa document.

In [7]:
kp_q = await stack.embedder.embed_query("who is eligible for the health card")
kp_hits = await stack.store.hybrid_search(ALICE, kp_q, filters=SearchFilters(jurisdictions=("kp",)), **SEARCH)
show_hits(kp_hits, "jurisdiction filter = kp")
assert kp_hits and {h.payload.jurisdiction for h in kp_hits} == {"kp"}
only_doc = await stack.store.hybrid_search(ALICE, kp_q, filters=SearchFilters(doc_ids=(doc_ids["docx"],)), **SEARCH)
assert {h.payload.doc_id for h in only_doc} == {doc_ids["docx"]}
print("doc_ids filter returns only", {h.payload.file_name for h in only_doc})

jurisdiction filter = kp
 #     rrf   dense  sparse  file                     text
 1  0.0328   0.604    3.88  sehat_card_faq.html      'Who is eligible? Every resident of Khyber Pakht…'
 2  0.0320   0.369    1.95  sehat_card_faq.html      'What is covered? The card covers hospitalisatio…'
 3  0.0318   0.165    1.97  sehat_card_faq.html      'Sehat Card Plus FAQ'
 4  0.0315   0.225    1.86  sehat_card_faq.html      '| Item | Limit | |---|---| | Yearly family limi…'
doc_ids filter returns only {'nadra_cnic_guide.docx'}


## 5. Tenant isolation
`u_bob` uploaded a private note. Every store call needs a `TenantContext`; Alice can read the public
corpus and her own uploads, never Bob's.

In [8]:
private_q = await stack.embedder.embed_query("urgent passport fee payments bank challan before 11 am")
alice_hits = await stack.store.hybrid_search(ALICE, private_q, filters=SearchFilters(), prefetch_k=30, limit=30)
bob_hits = await stack.store.hybrid_search(BOB, private_q, filters=SearchFilters(), prefetch_k=30, limit=30)
print("Alice sees owners:", sorted({h.owner_id for h in alice_hits}))
print("Bob   sees owners:", sorted({h.owner_id for h in bob_hits}))
print("Bob's top hit    :", bob_hits[0].payload.file_name)
assert all(h.owner_id != BOB.user_id for h in alice_hits), "private doc leaked to another user"
assert all(h.payload.file_name != "helpdesk_note.txt" for h in alice_hits)
assert bob_hits[0].payload.file_name == "helpdesk_note.txt" and bob_hits[0].owner_id == BOB.user_id
assert bob_hits[0].payload.injection_flag, "the injected instruction in the note is flagged at ingestion"
assert await stack.store.count(ALICE, doc_ids["private"]) == 0, "Alice cannot even count Bob's chunks"
assert await stack.store.count(BOB, doc_ids["private"]) > 0

Alice sees owners: ['u_public']
Bob   sees owners: ['u_bob', 'u_public']
Bob's top hit    : helpdesk_note.txt


## 6. Deleting a document removes its points

In [9]:
before = await stack.store.count(BOB, doc_ids["private"])
point_ids = await stack.store.point_ids(BOB, doc_ids["private"])
remaining = await stack.store.delete_document(BOB, doc_ids["private"])
after = await stack.store.count(BOB, doc_ids["private"])
print(f"points before={before} ({len(point_ids)} ids) -> remaining reported={remaining}, count after={after}")
assert before > 0 and remaining == 0 and after == 0
gone = await stack.store.hybrid_search(BOB, private_q, filters=SearchFilters(), prefetch_k=30, limit=30)
assert all(h.payload.doc_id != doc_ids["private"] for h in gone), "deleted chunks must not be searchable"
public_left = await stack.store.count(ALICE)
print("public chunks still indexed:", public_left)
assert public_left > 0
await stack.aclose()
print("\nAll embedding and retrieval checks passed.", "(offline fakes)" if stack.mode == "offline" else "(hosted models)")

points before=1 (1 ids) -> remaining reported=0, count after=0
public chunks still indexed: 20

All embedding and retrieval checks passed. (offline fakes)
